In [0]:
%run /Users/ishwargupta4554@gmail.com/Data_Quality/Data_Insertion_For_Inactivity

In [0]:
%sql

WITH FirstTransaction AS (
    SELECT
        CUSTOMERID,
        MIN(DATE) AS FirstTransactionDate
    FROM finance_tbl
    GROUP BY CUSTOMERID
),
RecentTransactions AS (
    SELECT DISTINCT CUSTOMERID
    FROM finance_tbl
    WHERE DATE >= CURRENT_DATE - INTERVAL '1 YEAR'
),
InactiveCustomers AS (
    SELECT
        c.CUSTOMERID,
        c.CREATEDAT,
        ft.FirstTransactionDate,
        CASE
            WHEN ft.FirstTransactionDate IS NULL THEN 'No Transactions'
            WHEN c.CREATEDAT < ft.FirstTransactionDate THEN 'Created Before First Transaction'
            ELSE 'Active'
        END AS InactivityReason
    FROM cust_tbl c
    LEFT JOIN FirstTransaction ft ON c.CUSTOMERID = ft.CUSTOMERID
    LEFT JOIN RecentTransactions rt ON c.CUSTOMERID = rt.CUSTOMERID
    WHERE rt.CUSTOMERID IS NULL -- No transactions in the last year
       OR ft.FirstTransactionDate IS NULL -- No transactions at all
       OR c.CREATEDAT < ft.FirstTransactionDate -- Created before first transaction
)
SELECT COUNT(*) AS inactive_customer_count
FROM InactiveCustomers
WHERE InactivityReason != 'Active';


inactive_customer_count
16


In [0]:
%sql

WITH cte1 AS (
    SELECT c.CUSTOMERID, MAX(f.date) AS max_date
    FROM cust_tbl AS c
    INNER JOIN finance_tbl AS f
    ON c.CUSTOMERID = f.CUSTOMERID
    GROUP BY c.CUSTOMERID
)

SELECT count(*) as inactiveCustomerCounts
FROM cte1
WHERE datediff(current_date(), max_date) > 100
-- and (select max(ct.CREATEDAT) from cust_tbl as ct where cte1.CUSTOMERID = ct.CUSTOMERID) > (select min(ft.DATE) from finance_tbl as ft where cte1.CUSTOMERID = ft.CUSTOMERID)

inactiveCustomerCounts
100


In [0]:
display(_sqldf)

inactiveCustomerCounts
100
